# Predictive AI Evaluation Challenge — Gated Residual Latent-Factor Model (Colab)

This notebook trains a **gated MLP residual** stacked on top of the metadata-only latent-factor model. Same data, same preprocessor, same 3-seed official-like validation, same submission packaging — only the model architecture changes. The original baseline notebook lives next to this one as `latent_factor_colab.ipynb`; you can run both back-to-back to do an ablation on the same split.

**Statistical form**
$$\eta_{m,b,c} = \underbrace{\mu + a_m + b_{b,c} + \frac{u_m \cdot v_{b,c}}{\sqrt{k}}}_{\text{base latent factor (unchanged)}} + s \cdot g(h_{m,b,c})$$
$$p_{m,b,c} = \sigma(\eta_{m,b,c})$$

where $s$ is a learned scalar (init **0**, so at $t=0$ the residual contributes exactly nothing and the strong baseline dominates), and $g(\cdot)$ is a SwiGLU-style block:

$$g(h) = W_{\text{down}}\bigl(\mathrm{SiLU}(W_{\text{gate}}\,\mathrm{LN}(h)) \odot W_{\text{up}}\,\mathrm{LN}(h)\bigr)$$

with input
$$h_{m,b,c} = [\,u_m\ \|\ v_{b,c}\ \|\ u_m\!\odot\! v_{b,c}\ \|\ |u_m - v_{b,c}|\ \|\ \mathrm{ModelTowerHidden}\ \|\ \mathrm{BenchTowerHidden}\,]$$

The model still **does not see `item_content`**. The baseline form is byte-identical to `latent_factor_colab.ipynb`'s model when the gated residual is disabled — same flags, same checkpoints would load. Headline number to beat: $\approx -0.5224$ (logistic baseline).

**Recommended runtime**: A100 (Runtime → Change runtime type → GPU → A100). L4/T4 also work; the residual block adds only a few tens of thousands of parameters.

## 1. Environment + GPU info

In [ ]:
import os, sys, subprocess, json, time, shutil
from pathlib import Path

print('Python:', sys.version.split()[0])
try:
    import torch
    print('torch:', torch.__version__, 'cuda:', torch.version.cuda, 'cuda_available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        free, total = torch.cuda.mem_get_info(0)
        print(f'GPU memory: free={free/1e9:.2f}GB total={total/1e9:.2f}GB')
except ImportError:
    print('torch is not installed yet — will install in next cell.')
subprocess.run(['nvidia-smi'], check=False)

## 2. Install dependencies

Colab already ships `torch`, `pandas`, `numpy`, `scikit-learn`, `pyarrow`. We only need to ensure `huggingface_hub` and `datasets` are present for the data download. We also install `tqdm` (already there but pinned for safety).

In [ ]:
%pip install -q --upgrade huggingface_hub datasets pyarrow pandas numpy scikit-learn tqdm

## 3. Clone the competition repo

The repo bundles `validation_harness/`, `starting_kit/Model_Info/model_info.csv`, `starting_kit/benchmark_info/benchmark_info.csv`, and `Google_Collab_harness/` (this folder). It does **not** include the response parquets — those come from HuggingFace in the next step.

In [ ]:
REPO_URL = 'https://github.com/bwathomas/Prediction-Competition-321M.git'
REPO_DIR = Path('/content/Prediction-Competition-321M').resolve()
if REPO_DIR.exists():
    print(f'{REPO_DIR} already exists, pulling latest …')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=False)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)

for sub in ['validation_harness', 'starting_kit/Model_Info', 'starting_kit/benchmark_info', 'Google_Collab_harness']:
    p = REPO_DIR / sub
    print(f'  {sub:40s} {"OK" if p.exists() else "MISSING"}')

GCH = REPO_DIR / 'Google_Collab_harness'
if str(GCH) not in sys.path:
    sys.path.insert(0, str(GCH))
os.chdir(REPO_DIR)
print('cwd:', os.getcwd())

## 4. Download the response parquets from HuggingFace

We download all `*.parquet` files from `aims-foundations/measurement-db` except the `*_traces.parquet` ones (different schema, not used). Total ≈ 1.5 GB.

In [ ]:
from huggingface_hub import HfApi, hf_hub_download

REPO_ID = 'aims-foundations/measurement-db'
DATA_DIR = REPO_DIR / 'starting_kit' / 'Data'
DATA_DIR.mkdir(parents=True, exist_ok=True)

api = HfApi()
files = api.list_repo_files(repo_id=REPO_ID, repo_type='dataset')
wanted = [
    f for f in files
    if f.endswith('.parquet') and not f.endswith('_traces.parquet')
]
print(f'{len(wanted)} parquets to download')

for f in wanted:
    out = DATA_DIR / Path(f).name
    if out.exists() and out.stat().st_size > 0:
        continue
    print(f'  downloading {f} …', flush=True)
    p = hf_hub_download(repo_id=REPO_ID, filename=f, repo_type='dataset', local_dir=str(DATA_DIR), local_dir_use_symlinks=False)
    if Path(p).resolve() != out.resolve():
        try:
            shutil.copy2(p, out)
        except shutil.SameFileError:
            pass
print('done. files:')
subprocess.run(['ls', '-lh', str(DATA_DIR)], check=False)

## 5. Build the official item-cold-start split

Reuses `validation_harness/scripts/prepare_split.py` so the validation we report is exactly the official-like protocol.

In [ ]:
HARNESS_DIR = REPO_DIR / 'validation_harness'
SPLITS_DIR = HARNESS_DIR / 'splits' / 'v1'
if not (SPLITS_DIR / 'train.parquet').exists():
    subprocess.check_call([
        sys.executable,
        str(HARNESS_DIR / 'scripts' / 'prepare_split.py'),
        '--data-dir', str(DATA_DIR),
        '--out-dir',  str(SPLITS_DIR),
        '--val-fraction', '0.10',
        '--seed', '0',
    ])
else:
    print(f'Reusing existing split at {SPLITS_DIR}')
subprocess.run(['ls', '-lh', str(SPLITS_DIR)], check=False)

## 6. Train the gated-residual latent-factor model + run official-like validation

Same training/validation pipeline as the baseline notebook, with the SwiGLU residual enabled via `--use-gated-residual`. Output is written to `outputs/latent_factor_gated/` so it does not overwrite a previously trained baseline (which lets section 8b do a proper ablation).

Defaults: `latent_dim=16`, `hidden_dim=256`, `num_layers=2`, `dropout=0.1`, `batch_size=65536`, `epochs=1000`, `patience=100`. Gated-residual defaults: `gated_hidden_dim=64`, `gated_dropout=0.05`, `residual_scale_init=0.0`, `learn_residual_scale=True`.

**Why those gated defaults?**
- `residual_scale_init=0.0` ensures that at $t=0$ the prediction equals the bare latent-factor base logit. The optimizer has to *grow* the residual scale to use the new MLP, so the strong baseline can never be wiped out by an unlucky initialization.
- `gated_hidden_dim=64` is enough to fit a few hundred (model, benchmark, condition) interaction patterns with negligible parameter overhead (~75K extra params on top of ~700K base).
- `gated_dropout=0.05` makes the residual easy to regularize even on tiny aggregated cells.

**Live progress reporting** (identical to the baseline notebook):

- A live `tqdm` bar tracks the planned **total training steps** with continuously updated **% done** and **ETA**.
- Every **10 optimizer steps** a one-line summary is printed: `step S/Total (X% done, ETA Y)  epoch=…  loss=…  elapsed=…`.
- Every **epoch** prints `train_loss`, `val_ll`, `val_brier`, `val_auc`, ETA, and — because the gated residual is on — the **current value of the learned residual scale**. Watch this number climb away from 0 if the MLP is finding signal; if it stays near 0 for many epochs the residual is being correctly identified as redundant.

In [ ]:
import os, subprocess

OUTPUT_DIR = REPO_DIR / 'outputs' / 'latent_factor_gated'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, '-u', str(GCH / 'run_latent_factor_colab.py'),
    '--data-dir',                str(DATA_DIR),
    '--splits-dir',              str(SPLITS_DIR),
    '--model-info-csv',          str(REPO_DIR / 'starting_kit' / 'Model_Info' / 'model_info.csv'),
    '--benchmark-info-csv',      str(REPO_DIR / 'starting_kit' / 'benchmark_info' / 'benchmark_info.csv'),
    '--validation-harness-dir',  str(HARNESS_DIR),
    '--output-dir',              str(OUTPUT_DIR),
    # --- base latent-factor hyperparameters (same defaults as baseline) ---
    '--latent-dim',  '16',
    '--hidden-dim',  '256',
    '--num-layers',  '2',
    '--dropout',     '0.1',
    '--lr',          '1e-3',
    '--weight-decay','1e-4',
    '--id-emb-l2',   '1e-3',
    '--batch-size',  '65536',
    '--epochs',      '1000',
    '--patience',    '100',
    # --- gated MLP residual on top of the base logit ---
    '--use-gated-residual',
    '--gated-hidden-dim',     '64',
    '--gated-dropout',        '0.05',
    '--residual-scale-init',  '0.0',
    '--learn-residual-scale',
    # --- live progress: per-step loss + tqdm-style "X% done, ETA Y" bar ---
    '--log-every-steps', '10',
    '--progress-bar',
    # --- official-like validation across 3 seeds ---
    '--official-seeds', '0', '1', '2',
    '--official-n', '5000',
    '--official-k', '5',
    '--logistic-baseline-ll', '-0.5224',
]


def _run_streaming(argv):
    """Run a child process and stream its stdout/stderr into the cell.

    Plain ``subprocess.run(cmd)`` lets the child write to the kernel's raw
    stdout/stderr file descriptors, which IPython does NOT forward to the
    cell display in most Colab/Jupyter setups -- so the cell looks empty
    even though the script printed plenty. We capture stdout (with stderr
    merged in) and re-print line-by-line so IPython's OutStream picks it up.
    """
    print('Running:', ' '.join(argv), flush=True)
    env = dict(os.environ, PYTHONUNBUFFERED='1', PYTHONIOENCODING='utf-8')
    proc = subprocess.Popen(
        argv,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,  # merge so order is preserved
        bufsize=1,                 # line-buffered
        text=True,
        env=env,
    )
    try:
        for line in proc.stdout:
            print(line, end='', flush=True)
    finally:
        proc.stdout.close()
    ret = proc.wait()
    if ret != 0:
        raise subprocess.CalledProcessError(ret, argv)


_run_streaming(cmd)

## 7. (Optional) sequential hyperparameter sweep with full progress

Runs the same script with `--sweep --use-gated-residual`, which samples `--sweep-budget` configs from a 7-dimensional grid (gated residual is on for **every** sampled config; the gated-specific dims `gated_hidden_dim`, `gated_dropout`, `residual_scale_init` are kept fixed at the defaults from section 6 for tractability — vary them by re-running this cell with different defaults if you want):

| dim | values |
| --- | --- |
| `latent_dim`   | 4, 8, 16, 32 |
| `hidden_dim`   | 128, 256 |
| `dropout`      | 0.05, 0.1, 0.2 |
| `weight_decay` | 1e-4, 1e-3 |
| `lr`           | 1e-3, 3e-3 |
| `id_emb_l2`    | 1e-4, 1e-3 |
| `patience`     | 5, 10 |

(`--sweep-mode full` would walk the entire 384-config grid; `random` picks a uniform subset of size `--sweep-budget`.)

**Why sequential here?** With `--parallel-runs 1` we get the *exact same* per-step progress as section 6 for **every** sweep run, one at a time. After each run finishes the next one starts and prints its own tqdm bar + per-step loss + current residual-scale value, so the cell output reads like a clean diary of "run 1 done → starting run 2 → ...". This is much easier to monitor than 8 interleaved tqdm bars on the same GPU. The shared preprocessor + tensor cache (built once in section 6) keeps per-run startup cheap.

If you'd rather trade observability for raw throughput, set `--parallel-runs 0` (auto) or e.g. `--parallel-runs 8` and the script will dispatch the configs through `ProcessPoolExecutor` with the `spawn` start method (tqdm bars are auto-disabled in that mode to avoid mangled output, but per-step + per-epoch text logs still go through).

In [ ]:
sweep_cmd = cmd + [
    '--sweep',
    '--sweep-mode', 'random',
    '--sweep-budget', '24',
    # Sequential: one process at a time so each run prints a clean tqdm bar +
    # per-10-step loss line, then we "pick up" the next run when it finishes.
    # Bump to e.g. 8 for parallel throughput (per-step bar is auto-disabled then).
    '--parallel-runs', '1',
    '--amp',
]
# NOTE: --use-gated-residual is already in `cmd` from section 6, so every
# sweep config inherits gated=True. The script's _build_full_grid uses
# **asdict(base) which propagates the gated fields verbatim.
_run_streaming(sweep_cmd)

## 8. Read the metrics + interpret results

In addition to the standard per-run / per-seed metrics, this cell also runs an **ablation comparison vs the baseline notebook** if you have already trained a model with `latent_factor_colab.ipynb` (which writes to `outputs/latent_factor/`). Two numbers are computed:

- the **gated minus baseline** delta on the **full validation set** (single deterministic LL — apples-to-apples, same eval rows)
- the **gated minus baseline** delta on the **3-seed official-like LL** (the number that determines the leaderboard)

If the gated residual is doing useful work, both deltas should be positive (more positive = better). If they hover near zero, the residual is correctly identifying that the latent-factor base captured everything; the trainable scale should also be near zero in that case (visible in the per-epoch log lines from section 6).

In [ ]:
import pandas as pd, json
summary = json.loads((OUTPUT_DIR / 'metrics.json').read_text())
print(json.dumps(summary, indent=2))

print('\nbaseline_comparison.csv:')
print(pd.read_csv(OUTPUT_DIR / 'baseline_comparison.csv').to_string(index=False))

print('\nofficial_seeds.csv:')
print(pd.read_csv(OUTPUT_DIR / 'official_seeds.csv').to_string(index=False))

if (OUTPUT_DIR / 'runs.csv').exists():
    print('\nruns.csv (top 10 by final_val_log_likelihood):')
    runs = pd.read_csv(OUTPUT_DIR / 'runs.csv')
    cols = ['run_idx', 'latent_dim', 'weight_decay', 'dropout', 'best_epoch',
            'use_gated_residual', 'gated_hidden_dim', 'residual_scale_init',
            'final_train_log_likelihood', 'final_val_log_likelihood', 'wall_seconds']
    cols = [c for c in cols if c in runs.columns]
    print(runs[cols].sort_values('final_val_log_likelihood', ascending=False).head(10).to_string(index=False))

off_mean = summary.get('official_mean_log_likelihood')
imp = summary.get('improvement_vs_logistic_baseline')
if off_mean is not None and imp is not None:
    verdict = 'BEATS' if (imp or 0) > 0 else 'TRAILS'
    print(f'\nOfficial-like mean LL = {off_mean:+.4f}    {verdict} the logistic baseline by {imp:+.4f}')

# --------------------------------------------------------------------------
# Ablation vs the baseline notebook (outputs/latent_factor/), if it exists.
# Apples-to-apples: same split, same val rows, same official seeds.
# --------------------------------------------------------------------------
BASELINE_DIR = REPO_DIR / 'outputs' / 'latent_factor'
if (BASELINE_DIR / 'metrics.json').exists():
    print('\n' + '=' * 72)
    print('ABLATION vs baseline (latent_factor_colab.ipynb output)')
    print('=' * 72)
    base_summary = json.loads((BASELINE_DIR / 'metrics.json').read_text())

    base_full_val = base_summary.get('best_run', {}).get('final_val_log_likelihood', float('nan'))
    gated_full_val = summary.get('best_run', {}).get('final_val_log_likelihood', float('nan'))
    base_off = base_summary.get('official_mean_log_likelihood', float('nan'))
    gated_off = summary.get('official_mean_log_likelihood', float('nan'))

    rows = pd.DataFrame([
        {
            'metric': 'full-val log-likelihood',
            'baseline':  base_full_val,
            'gated':     gated_full_val,
            'delta':     (gated_full_val - base_full_val)
                          if all(v == v for v in (base_full_val, gated_full_val)) else float('nan'),
        },
        {
            'metric': 'official 3-seed mean LL',
            'baseline':  base_off,
            'gated':     gated_off,
            'delta':     (gated_off - base_off)
                          if all(v == v for v in (base_off, gated_off)) else float('nan'),
        },
    ])
    print(rows.to_string(index=False, float_format=lambda x: f'{x:+.4f}'))
    delta = rows.iloc[1]['delta']
    if delta == delta:  # not NaN
        verdict = 'BEATS' if delta > 0 else ('TIES' if abs(delta) < 1e-4 else 'TRAILS')
        print(f"\nGated residual {verdict} the baseline by {delta:+.4f} on the official metric.")
else:
    print('\n(no outputs/latent_factor/metrics.json yet -- run latent_factor_colab.ipynb to enable ablation)')

## 9. Download the gated model

Builds **two** zip archives from `outputs/latent_factor_gated/` and triggers browser downloads via `google.colab.files`:

1. **`codabench_gated_submission.zip`** — slim, **upload-ready for Codabench**. Files at the top level of the zip (`model.py`, `latent_factor_pytorch.py`, `best_model.pt`, `preprocessor.pkl`, `model_info.csv`, `benchmark_info.csv`). The bundled `best_model.pt` includes the saved `LFConfig` with `use_gated_residual=True`, so when Codabench imports `model.py` and that calls `load_artifacts(...)`, the model is reconstructed with the gated residual block intact. Same `predict()` contract as the baseline submission — Codabench cannot tell which model variant it received.
2. **`latent_factor_gated_submission.zip`** — archival bundle (everything above + `weights.pt` + `metrics.json` + `runs.csv` + `baseline_comparison.csv` + `official_seeds.csv` + `reproduce.json`/`.sh`/`.bat`). Reload elsewhere with:

   ```python
   from latent_factor_pytorch import load_artifacts
   model, preprocessor, config = load_artifacts('latent_factor_gated_submission/')
   assert config.use_gated_residual is True
   ```

Outside Colab the cell prints absolute paths so you can copy the zips manually.

In [ ]:
import zipfile
from pathlib import Path

# Files that MUST live at the top level of the Codabench-bound zip so the
# platform's loader can find model.py and the model's runtime deps.
CODABENCH_REQUIRED = [
    'model.py',                  # entry point with predict()
    'latent_factor_pytorch.py',  # model + preprocessor + inference classes (gated-aware)
    'best_model.pt',             # full bundle: state_dict + config_dict (use_gated_residual=True)
    'preprocessor.pkl',          # fitted preprocessor (vocabs, scalers, lookups)
    'model_info.csv',            # baked metadata read at load time
    'benchmark_info.csv',        # baked metadata read at load time
]

# Extra files included in the archival bundle but NOT in the Codabench zip.
ARCHIVAL_EXTRAS = [
    'weights.pt',
    'metrics.json',
    'baseline_comparison.csv',
    'official_seeds.csv',
    'runs.csv',
    'reproduce.json',
    'reproduce.sh',
    'reproduce.bat',
]

# Resolve OUTPUT_DIR if it wasn't already defined (e.g. running this cell standalone)
try:
    OUTPUT_DIR
except NameError:
    OUTPUT_DIR = Path('/content/Prediction-Competition-321M/outputs/latent_factor_gated').resolve()

OUTPUT_DIR = Path(OUTPUT_DIR).resolve()
SUBMISSION_DIR = OUTPUT_DIR / 'submission'
CODABENCH_ZIP = OUTPUT_DIR / 'codabench_gated_submission.zip'
ARCHIVE_ZIP = OUTPUT_DIR / 'latent_factor_gated_submission.zip'

print(f'Bundling artifacts under: {OUTPUT_DIR}')
if not OUTPUT_DIR.exists():
    raise SystemExit(f'OUTPUT_DIR does not exist: {OUTPUT_DIR}. Run section 6 first.')
if not SUBMISSION_DIR.exists():
    raise SystemExit(
        f'submission/ folder missing under {OUTPUT_DIR}. '
        f'It is built by run_latent_factor_colab.py at the end of section 6.'
    )

# Sanity-check that we are actually shipping the gated variant.
import json as _json
_metrics = _json.loads((OUTPUT_DIR / 'metrics.json').read_text()) if (OUTPUT_DIR / 'metrics.json').exists() else {}
_best = _metrics.get('best_run', {})
if 'use_gated_residual' in _best:
    print(f'  best_run.use_gated_residual = {_best["use_gated_residual"]}')
    if not _best['use_gated_residual']:
        print('  WARNING: best_run did NOT have the gated residual enabled. '
              'Re-run section 6 with --use-gated-residual.')

# ---------------------------------------------------------------------------
# 1. Codabench-ready zip: files at the TOP LEVEL of the archive.
# ---------------------------------------------------------------------------
print(f'\n[1/2] Building Codabench-ready zip -> {CODABENCH_ZIP}')
missing = []
with zipfile.ZipFile(CODABENCH_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for fname in CODABENCH_REQUIRED:
        src = SUBMISSION_DIR / fname
        if not src.exists():
            missing.append(fname)
            continue
        zf.write(src, arcname=fname)
        print(f'    + {fname}  ({src.stat().st_size / 1024:.1f} KB)')
if missing:
    raise SystemExit(
        f'Codabench bundle missing required files: {missing}. '
        f'Re-run section 6 to rebuild submission/.'
    )
print(f'    => {CODABENCH_ZIP.name}  ({CODABENCH_ZIP.stat().st_size / 1e6:.2f} MB)')

# ---------------------------------------------------------------------------
# 2. Archival bundle.
# ---------------------------------------------------------------------------
print(f'\n[2/2] Building archival bundle -> {ARCHIVE_ZIP}')
with zipfile.ZipFile(ARCHIVE_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    bundled, skipped = [], []
    for fname in CODABENCH_REQUIRED + ARCHIVAL_EXTRAS:
        src = OUTPUT_DIR / fname
        if not src.exists():
            src = SUBMISSION_DIR / fname
        if src.exists():
            zf.write(src, arcname=f'latent_factor_gated_submission/{fname}')
            bundled.append(fname)
        else:
            skipped.append(fname)
    for p in SUBMISSION_DIR.rglob('*'):
        if p.is_file() and '__pycache__' not in p.parts:
            arc = Path('latent_factor_gated_submission/submission') / p.relative_to(SUBMISSION_DIR)
            zf.write(p, arcname=str(arc).replace('\\', '/'))
print(f'    bundled : {bundled}')
if skipped:
    print(f'    skipped : {skipped}  (not produced; ok if section 6 ran without --sweep)')
print(f'    => {ARCHIVE_ZIP.name}  ({ARCHIVE_ZIP.stat().st_size / 1e6:.2f} MB)')

# ---------------------------------------------------------------------------
# 3. Trigger browser downloads in Colab; gracefully degrade elsewhere.
# ---------------------------------------------------------------------------
try:
    from google.colab import files as _colab_files  # type: ignore
    print(f'\nTriggering download of {CODABENCH_ZIP.name} (Codabench-ready) ...')
    _colab_files.download(str(CODABENCH_ZIP))
    print(f'Triggering download of {ARCHIVE_ZIP.name} (archival) ...')
    _colab_files.download(str(ARCHIVE_ZIP))
except ImportError:
    print('\nNot running in Colab — no browser download triggered.')
    print(f'  Codabench zip: {CODABENCH_ZIP}')
    print(f'  Archive  zip: {ARCHIVE_ZIP}')
except Exception as e:
    print(f'\ngoogle.colab.files.download failed: {type(e).__name__}: {e}')
    print(f'Manual fallback paths:')
    print(f'  Codabench zip: {CODABENCH_ZIP}')
    print(f'  Archive  zip: {ARCHIVE_ZIP}')

## 10. Try the gated model in the actual competition

The downloaded `codabench_gated_submission.zip` is the file you upload to Codabench. Concrete steps:

1. **Verify the bundle locally first** (next cell). The verification additionally asserts that the bundled checkpoint's config has `use_gated_residual=True`, and that `LatentFactorModel` rebuilt from the saved weights has both a `gated_residual` submodule and a `residual_scale_raw` parameter/buffer — i.e., we are actually shipping the gated variant.
2. **Upload to Codabench**. Open the competition page → "Submit / View Results" tab → "Submit a new entry" → select `codabench_gated_submission.zip`. Codabench unzips it and runs `model.py`. From the platform's perspective the gated model is indistinguishable from the baseline submission — same `predict(input, labeled=None) -> float` contract — so no platform-side configuration changes are needed.
3. **Wait for scoring**. Same hidden test slice (5000 items, K=5 adaptive labels per category). Module-level init now reconstructs the gated residual block from the saved config; the (~75K extra) parameters add negligible startup cost.
4. **Watch the leaderboard** for the submission's mean log-likelihood. Compare against:
   - the logistic baseline reference: `−0.5224`
   - the baseline notebook's leaderboard score (if you submitted it earlier — that is the apples-to-apples comparison for whether the gated residual actually helps)
   - the section 8b ablation table from this notebook (your own held-out estimate of the gated lift)

**Constraints to remember** (same as for the baseline submission):

- Up to **50 scored submissions per team per UTC day**. Use one slot for the baseline and one for this gated variant if you want a real apples-to-apples leaderboard delta.
- No outbound internet. Bundle is self-contained.
- Module-level code runs once; per-call work is a cache hit (the gated residual is computed once per `(benchmark, condition, model_name)` and memoized).
- We still ship neither a `models.txt` nor a `labeling.py`; the same defaults from the baseline submission apply.

In [ ]:
# Verify codabench_gated_submission.zip end-to-end BEFORE uploading.
# Unzips to a fresh temp dir, imports model.py from there, and runs predict()
# on a few representative inputs. Additional checks vs the baseline cell:
#   * the saved LFConfig must have use_gated_residual = True
#   * the rebuilt LatentFactorModel must own a `gated_residual` submodule
#   * `residual_scale_raw` must exist and be a finite float
import importlib, importlib.util, sys, tempfile, zipfile, time, math
from pathlib import Path

assert CODABENCH_ZIP.exists(), f'Run section 9 first — {CODABENCH_ZIP} not found.'

with tempfile.TemporaryDirectory() as td:
    extract_dir = Path(td) / 'submission'
    extract_dir.mkdir()
    print(f'Unzipping {CODABENCH_ZIP.name} -> {extract_dir} ...')
    with zipfile.ZipFile(CODABENCH_ZIP, 'r') as zf:
        zf.extractall(extract_dir)
    extracted = sorted(p.name for p in extract_dir.iterdir())
    print(f'  extracted files: {extracted}')

    # Make sure we import the *bundled* model.py, not any cached copy.
    for mod_name in ('model', 'latent_factor_pytorch'):
        sys.modules.pop(mod_name, None)
    sys.path.insert(0, str(extract_dir))
    try:
        t_init = time.perf_counter()
        spec = importlib.util.spec_from_file_location('model', extract_dir / 'model.py')
        model_mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(model_mod)
        init_dt = time.perf_counter() - t_init
        print(f'  module init OK in {init_dt*1000:.1f} ms (loads weights + preprocessor)')

        # ---- Gated-specific assertions ----------------------------------
        cfg = model_mod._CONFIG
        inner = model_mod._MODEL
        assert getattr(cfg, 'use_gated_residual', False) is True, \
            'LFConfig.use_gated_residual is False -- you packaged the BASELINE bundle by mistake.'
        assert hasattr(inner, 'gated_residual'), \
            'LatentFactorModel has no gated_residual submodule despite use_gated_residual=True.'
        assert hasattr(inner, 'residual_scale_raw'), \
            'LatentFactorModel has no residual_scale_raw attribute.'
        scale_now = float(inner.residual_scale_raw.detach().item())
        assert math.isfinite(scale_now), f'residual_scale_raw is not finite: {scale_now}'
        n_resid = inner.count_residual_params()
        n_total = sum(p.numel() for p in inner.parameters())
        print(f'  gated config: hidden_dim={cfg.gated_hidden_dim} dropout={cfg.gated_dropout} '
              f'scale_init={cfg.residual_scale_init} learn_scale={cfg.learn_residual_scale}')
        print(f'  residual params: {n_resid}/{n_total} ({100*n_resid/max(1,n_total):.2f}%)  '
              f'scale_now={scale_now:+.4f}')

        # A few representative inputs.
        sample_inputs = [
            {'benchmark': 'mmlupro',    'condition': 'zero-shot',
             'subject_content': 'Name: gpt-4o\nOrganization: OpenAI\nFamily: gpt-4o',
             'item_content': 'What is 2 + 2?'},
            {'benchmark': 'ai2d_test',  'condition': 'none',
             'subject_content': 'Name: claude-3-5-sonnet-20240620',
             'item_content': 'Describe the diagram.'},
            {'benchmark': 'gsm8k',      'condition': '',
             'subject_content': 'Name: __nonexistent_model_for_unk_test__',
             'item_content': 'A train leaves Boston ...'},
            {'benchmark': '__unknown_benchmark__', 'condition': 'none',
             'subject_content': '',
             'item_content': ''},
        ]
        print('\nTrying predict() on sample inputs:')
        t_pred = time.perf_counter()
        for i, x in enumerate(sample_inputs, start=1):
            p = model_mod.predict(x, labeled=None)
            ok = isinstance(p, float) and math.isfinite(p) and 0.0 <= p <= 1.0
            status = 'OK' if ok else 'FAIL'
            print(f'  [{status}] sample {i}: bench={x["benchmark"]!r:20s} '
                  f'cond={x["condition"]!r:14s} subj={x["subject_content"][:40]!r:42s} -> p={p:.4f}')
            assert ok, f'predict() returned invalid value for sample {i}: {p!r}'
        pred_dt = time.perf_counter() - t_pred
        print(f'\n4 calls in {pred_dt*1000:.1f} ms total ({pred_dt*1000/4:.2f} ms/call avg, includes cache misses)')
    finally:
        sys.path.remove(str(extract_dir))
        for mod_name in ('model', 'latent_factor_pytorch'):
            sys.modules.pop(mod_name, None)

print('\n=== Gated bundle is upload-ready. ===')
print(f'Upload this file to Codabench: {CODABENCH_ZIP}')
print(f'  size: {CODABENCH_ZIP.stat().st_size / 1e6:.2f} MB')